# Benchmarking Titans-augmented Gemma inside RLM

This notebook runs the four bench scripts shipped in
`scripts/benchmarks/`:

| Bench | Capability tested |
|-------|-------------------|
| `bench_reasoning.py`    | multi-step math reasoning (GSM8K-style)   |
| `bench_tool_use.py`     | agentic tool calling                       |
| `bench_codegen.py`      | code generation (HumanEval-style)          |
| `bench_long_context.py` | long-context needle-in-haystack            |

We compare two backends:

1. The vanilla API baseline (e.g. `gpt-5-nano`).
2. Local Gemma 3 1B-IT augmented with a Titans memory module.

A Colab T4 is enough; expect ~10 minutes for `--num-samples 5`.

In [ ]:
%pip install --quiet "rlms[titans]" || %pip install --quiet -e /content/rlm
import os
if not os.path.exists("/content/rlm"):
    !git clone --depth 1 https://github.com/alexzhang13/rlm /content/rlm 2>/dev/null || true


In [ ]:
# Optional - set this if you also want to run the OpenAI baseline:
# os.environ["OPENAI_API_KEY"] = "sk-..."


## Run all four benchmarks against Gemma + Titans

In [ ]:
!cd /content/rlm && python -m scripts.benchmarks.run_all \
    --backend titans_hf --model google/gemma-3-1b-it \
    --num-samples 3 --max-iterations 6 --output-dir bench_results/titans


## Run the same benches against an API baseline (optional)

In [ ]:
# !cd /content/rlm && python -m scripts.benchmarks.run_all \
#     --backend openai --model gpt-5-nano \
#     --num-samples 3 --max-iterations 6 --output-dir bench_results/openai


## Sweep retention surrogates (Miras)

`MirasMemory` exposes the inner-loop retention loss as a config knob.
Each variant captures a different attention/memory style:

* `l2` — vanilla Titans
* `l1` — sign-SGD style updates (mamba-like)
* `huber` — smooth interpolation
* `kl` — softmax / Bregman retention (closer to attention)


In [ ]:
!for r in l2 l1 huber kl; do \
    cd /content/rlm && python -m scripts.benchmarks.run_all \
        --backend titans_hf --memory-flavor miras --retention $r \
        --model google/gemma-3-1b-it \
        --num-samples 2 --max-iterations 4 \
        --skip codegen --output-dir bench_results/miras-$r ;\
done


## Aggregate & plot

In [ ]:
import json, glob
import pandas as pd
rows = []
for d in glob.glob("/content/rlm/bench_results/**/summary.json", recursive=True):
    with open(d) as f:
        s = json.load(f)
    for bench, val in s.get("benches", {}).items():
        if "error" in val: continue
        rows.append(
            {
                "config": d.split("/")[-2],
                "bench": bench,
                "accuracy": val["accuracy"],
                "n_examples": val["n_examples"],
                "n_errors": val["n_errors"],
                "time_s": val["total_time_seconds"],
            }
        )
df = pd.DataFrame(rows)
df.pivot_table(index="config", columns="bench", values="accuracy")


### Reading the table

You should see Titans-augmented Gemma score competitively with the API
baseline on long-context needle-in-haystack (the memory's strong suit)
while being below the baseline on reasoning / codegen (where Gemma 1B
alone isn't strong).  Sweeping the Miras retention surrogate gives you
a sense of which inner-loop optimisation problem fits each task best.

For longer experiments, raise `--num-samples` and run on a bigger
backbone (`google/gemma-2-9b-it`).